# 74 ? Experimento TF-IDF + HDBSCAN

Este notebook agrupa descri??es normalizadas de itens de supermercado usando TF-IDF e HDBSCAN. A an?lise usa dist?ncia cosseno, n?o usa NCM nem r?tulos humanos e trata ru?do como resultado expl?cito do algoritmo. A amostra configur?vel mant?m o custo do experimento controlado. Execute de cima para baixo em uma sess?o limpa.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook74")
pd.set_option("display.max_colwidth", 140)


## 1. Par?metros

`min_cluster_size` define o menor grupo que o HDBSCAN pode selecionar; `min_samples` controla o qu?o conservadora ? a estimativa de densidade. Quando `min_samples=None`, o algoritmo usa `min_cluster_size`. A sele??o `eom` procura grupos persistentes, enquanto `leaf` tende a produzir grupos menores e mais espec?ficos. A grade ? explorat?ria; silhouette e cobertura n?o medem acur?cia sem?ntica.

A an?lise cruza as sementes de amostragem com os mesmos analyzers do notebook 72. Para isolar os efeitos, cada analyzer usa a mesma amostra dentro de uma semente. `word` usa n-gramas de palavras e `char_wb` usa n-gramas de caracteres.

O HDBSCAN do scikit-learn n?o aceita pesos por observa??o. Cada descri??o ?nica contribui uma vez para o agrupamento; a frequ?ncia permanece dispon?vel para ordenar os clusters e as descri??es nos relat?rios. A silhouette ? calculada em uma amostra reprodut?vel dos itens n?o classificados como ru?do para limitar o custo.


In [ ]:
DATASET_NAME = "2025_NCMs_filtrados-sem-duplicatas"  # Nome da pasta em data/processed; altere apenas este valor para trocar a base.

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("N?o encontrei a raiz do reposit?rio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/per?odo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve come?ar com '{DATASET_NAME}-' para derivar o per?odo: "
        f"{DATASET_PATH.name}"
    )

PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MAX_DESCRIPTIONS = 5_000  # Amostra reprodut?vel para limitar o custo do experimento.
SAMPLE_SEEDS = [42]
ANALYZER_CONFIGS = {
    "char_wb_3_5": ("char_wb", (3, 5)),
    "char_wb_4_8": ("char_wb", (4, 8)),
    "char_wb_5_10": ("char_wb", (5, 10)),
    "char_wb_1_15": ("char_wb", (1, 15)),
    "word_1_2": ("word", (1, 2)),
    "word_1_3": ("word", (1, 3)),
    "word_1_4": ("word", (1, 4)),
    "word_1_5": ("word", (1, 5)),
    "word_2_4": ("word", (2, 4)),
    "word_2_5": ("word", (2, 5)),
}
MIN_DF = 10
MAX_FEATURES = 100_000
MIN_CLUSTER_SIZE_VALUES = [5, 10, 20]
MIN_SAMPLES_VALUES = [None, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"  # Alternativa: "leaf".
HDBSCAN_N_JOBS = 1
SILHOUETTE_SAMPLE_SIZE = 1_000
OUTPUT_ROOT = Path("reports/tfidf_hdbscan")

TSNE_MAX_POINTS = 5_000  # Amostra fixa para limitar o custo da proje??o.
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_SVD_COMPONENTS = 50  # Redu??o pr?via para matrizes TF-IDF esparsas.


### Configura??o do ranking final


In [ ]:
# O limite filtra apenas o ranking e a sele??o final; n?o altera os ajustes do HDBSCAN.
MAX_CLUSTERS_FOR_RANKING = 200

# Os pesos definem a contribui??o relativa de silhouette e cobertura (1 - ru?do).
SILHOUETTE_WEIGHT = 0.5
NOISE_WEIGHT = 0.5


## 2. Fun??es da an?lise

A prepara??o remove descri??es vazias, mant?m uma linha por descri??o e valida que a frequ?ncia total n?o foi alterada. Cada configura??o ? executada sobre a mesma matriz TF-IDF da respectiva representa??o.


In [ ]:
def prepare_items(frame):
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels, sample_size, random_state):
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_positions = np.flatnonzero(~noise_mask)
    clustered_labels = labels[clustered_positions]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    silhouette_count = min(sample_size, len(clustered_labels))
    if silhouette_count:
        rng = np.random.default_rng(random_state)
        sample_positions = np.sort(rng.choice(
            clustered_positions, size=silhouette_count, replace=False
        ))
        sample_labels = labels[sample_positions]
        sample_cluster_count = len(np.unique(sample_labels))
        if 2 <= sample_cluster_count < silhouette_count:
            score = float(silhouette_score(matrix[sample_positions], sample_labels, metric="cosine"))
        else:
            silhouette_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_sample_size": silhouette_count,
        "silhouette_cosine_without_noise": score,
    }

def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state):
    """Calcula uma proje??o t-SNE reprodut?vel sobre uma amostra das linhas."""
    point_count = features.shape[0]
    if point_count < 3:
        raise ValueError("O t-SNE requer pelo menos tr?s descri??es.")

    sample_count = min(point_count, max_points)
    if sample_count < 3:
        raise ValueError("TSNE_MAX_POINTS deve ser pelo menos tres.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled_features = features[indices]

    if sampled_features.shape[1] > 1:
        component_count = min(TSNE_SVD_COMPONENTS, sampled_features.shape[1] - 1, sample_count - 1)
        reduced_features = TruncatedSVD(
            n_components=component_count, random_state=random_state,
        ).fit_transform(sampled_features)
    else:
        reduced_features = sampled_features.toarray()

    effective_perplexity = min(float(perplexity), (sample_count - 1) / 3)
    coordinates = TSNE(
        n_components=2,
        perplexity=effective_perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=max_iter,
        random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced_features)
    return indices, coordinates, reduced_features.shape[1]


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ru?do em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descri??es como ru?do"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ru?do em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de prepara??o de dados do projeto, com descri??es de supermercado, NCMs e duplicatas agregadas por descri??o. A amostragem ? feita uma vez por semente e reutilizada entre os analyzers daquela semente.


In [ ]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))


## 4. Vetorizar, ajustar HDBSCAN e comparar par?metros

Cada combina??o de semente e analyzer gera uma matriz TF-IDF pr?pria. `metric="cosine"` trabalha diretamente com a matriz esparsa. O ru?do (`cluster = -1`) ? exclu?do da silhouette e contabilizado separadamente; configura??es com menos de dois clusters v?lidos ficam sem silhouette. Para controlar o custo, a silhouette usa no m?ximo `SILHOUETTE_SAMPLE_SIZE` itens n?o classificados como ru?do.

Para visualiza??o, uma amostra fixa de at? `TSNE_MAX_POINTS` descri??es ? reduzida com TruncatedSVD e projetada uma vez por representa??o. As coordenadas s?o reutilizadas para salvar um PNG por configura??o.


In [ ]:
records = []
labels_by_run = {}
items_by_run = {}
matrices_by_representation = {}
items_by_seed = {}

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
        logger.info(
            "Vetorizando analyzer=%s, seed=%d, descri??es=%d",
            analyzer, sample_seed, len(sampled_items),
        )
        vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, min_df=MIN_DF,
            max_features=MAX_FEATURES, sublinear_tf=True, norm="l2",
        )
        matrix = vectorizer.fit_transform(sampled_items[DESCRIPTION_NORMALIZED].astype(str))
        zero_vector_count = int((matrix.getnnz(axis=1) == 0).sum())
        matrices_by_representation[(sample_seed, representation)] = matrix
        logger.info(
            "Analyzer=%s seed=%d: %d termos, %d vetores zerados",
            analyzer, sample_seed, matrix.shape[1], zero_vector_count,
        )

        for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
            for min_samples in MIN_SAMPLES_VALUES:
                min_samples_label = "default" if min_samples is None else str(min_samples)
                run_id = (
                    f"seed{sample_seed}_{representation}_min_cluster_size{min_cluster_size}"
                    f"_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}"
                )
                logger.info("Ajustando HDBSCAN %s", run_id)
                model = HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric="cosine",
                    algorithm="brute",
                    cluster_selection_method=CLUSTER_SELECTION_METHOD,
                    n_jobs=HDBSCAN_N_JOBS,
                    copy=True,
                )
                labels = model.fit_predict(matrix)
                metrics = score_configuration(
                    matrix, labels, SILHOUETTE_SAMPLE_SIZE,
                    TSNE_RANDOM_STATE + sample_seed,
                )
                records.append({
                    "run_id": run_id, "sample_seed": sample_seed,
                    "representation": representation, "analyzer": analyzer,
                    "ngram_range": str(ngram_range),
                    "min_cluster_size": min_cluster_size,
                    "min_samples": min_samples,
                    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                    "frequency_weighted": False, **metrics,
                })
                labels_by_run[run_id] = labels
                items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)


### Calcular a metrica composta e ordenar os resultados

A silhouette e reescalada para 0 a 1 e combinada com a proporcao de descricoes classificadas (`1 - noise_fraction`). O score composto determina a ordem; silhouette e ruido servem apenas de desempate quando os scores forem iguais.


In [ ]:
if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].le(MAX_CLUSTERS_FOR_RANKING)
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        "Nenhum resultado atende ao limite de "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste min_cluster_size/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))


In [ ]:
display(ranking)

In [ ]:
ranking.head(20)

## 5. Revisar um resultado e salvar relat?rios

Escolha `SELECTED_RUN_ID` ap?s comparar quantidade de clusters, ru?do e silhouette. A tabela de resumo inclui `-1` como grupo de ru?do. Os arquivos s?o gravados em uma pasta com identificador UTC ?nico para preservar execu??es anteriores.

Os gr?ficos ficam em `tsne_plots/`, um PNG por configura??o, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execu??o.


In [ ]:
SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

## 6. Gerar visualiza??es t-SNE

Esta etapa roda depois do ajuste e do ranking. Ela projeta as matrizes TF-IDF j? calculadas e salva as coordenadas e os gr?ficos, sem executar novamente o HDBSCAN.


In [ ]:
# As projecoes sao calculadas depois do treinamento para separar visualizacao e ajuste do HDBSCAN.
tsne_by_representation = {}
for (sample_seed, representation), matrix in matrices_by_representation.items():
    logger.info("Calculando t-SNE para %s, seed=%d", representation, sample_seed)
    tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + sample_seed,
    )
    tsne_by_representation[(sample_seed, representation)] = {
        "indices": tsne_indices, "coordinates": tsne_coordinates,
        "reduced_dimensions": tsne_dimensions,
    }

# Salva uma tabela de coordenadas por representacao e um PNG para cada ajuste de HDBSCAN.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    projection = tsne_by_representation[(int(result_row["sample_seed"]), result_row["representation"])]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

logger.info("Imagens t-SNE salvas: %d", len(labels_by_run))
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": False,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "analyzer_configs": {
        representation: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "min_df": MIN_DF, "max_features": MAX_FEATURES,
    "selected_run_id": SELECTED_RUN_ID,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS, "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    __import__("json").dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


## 7. Interpreta??o

- `noise_fraction` alta significa que muitas descri??es n?o entraram em grupos densos para os par?metros escolhidos.
- `min_cluster_size` controla o menor grupo selecion?vel. `min_samples` maior tende a exigir evid?ncia local mais forte e pode aumentar o ru?do.
- Silhouette mais alta deve ser lida junto da fra??o de ru?do e do tamanho da amostra usada para calcul?-la; uma pontua??o calculada em poucos itens pode ser enganosa.
- HDBSCAN n?o atribui automaticamente novos produtos a clusters neste fluxo e n?o define categorias sem revis?o humana.
